# Sicilian–English–Italian Neural Machine Translation
### A reproducible, end-to-end pipeline

This notebook reproduces our whole model from start to finish, structured like a short
paper. Each section explains *what* and *why* (as in the slides), then runs the code — the
implementations live in the repo's `.py` files (`experiments/nllb/nllb_pipeline.py`,
`experiments/dataset/normalize_scn.py`), the notebook only orchestrates.

**Runtime → Change runtime type → GPU.**

## 1. Introduction

Sicilian is a low-resource Romance language, absent from the major MT services. Wdowiak's
*Tradutturi Sicilianu* is the published state of the art, built by training a model from
scratch on tens of millions of pairs (his "reverse-training" recipe).

**Research question.** Can a *pretrained* massively-multilingual model (NLLB-200), adapted
with parameter-efficient fine-tuning, reach that quality? And where it does not, is the gap
due to **method** or to **data**?

**Approach.** We (i) assemble a reproducible corpus, (ii) normalize to Standard Sicilian,
(iii) fine-tune NLLB-200 with LoRA, (iv) evaluate on a frozen literary test set, and
(v) apply the feasible parts of his recipe (multilingual, back-translation) plus a
normalization ablation, to attribute the residual gap to method vs data.

## 2. Setup
Clone the repo for the `.py` code, install dependencies, mount Drive.

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao
import os, sys, json
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
sys.path += ['SicilianNMT/experiments/nllb', 'SicilianNMT/experiments/dataset']
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'; DATA = f'{OUT}/data'
def read(p): return open(p, encoding='utf-8').read().splitlines()
from nllb_pipeline import load_base, attach_lora, build_dataset, finetune, translate, score
from normalize_scn import normalize
print('ready')

## 3. Data

### 3.1 Sources, extraction & alignment
The parallel text is built automatically: extract the *Arba Sicula* PDFs with PyMuPDF,
classify each page's language, confirm facing pairs and align sentences with **LaBSE**
embeddings + a monotonic dynamic program (no dictionary, no hunalign). We then add the
curated Napizia NLLB / WikiMatrix datasets and deduplicate. Code:
`experiments/extraction/build_all.py` and `experiments/dataset/assemble.py` (they require the
PDFs and the public datasets). A 1k/1k valid/test split is **frozen** so every result stays
comparable.

### 3.2 Loading the prepared dataset

In [ ]:
train = {l: read(f'{DATA}/train.{l}') for l in ('scn', 'en')}
test  = {l: read(f'{DATA}/test.{l}')  for l in ('scn', 'en')}
print('train', len(train['scn']), '| test', len(test['scn']))
print('sample pair:\n  scn:', train['scn'][0], '\n  en :', train['en'][0])

## 4. Preprocessing

### 4.1 Standardization (Standard Sicilian)
Sicilian has no single official orthography (many spellings, omitted accents). We normalize
toward **Standard Sicilian** — Cipolla's standard, augmented by Wdowiak for MT clarity — with
`normalize_scn.py`. The light **std** level applies spelling rules (*cchiù→chiù*,
*io/iu/eu→jo*, *non→nun*, *çiuri→ciuri*) and keeps accents/contractions; the **full** level
additionally uncontracts (*dû→di lu*) and ASCII-folds. We use **std** (its effect is measured
in the ablation, §7.3).

In [ ]:
print('before:', test['scn'][0])
for d in (train, test):
    d['scn'] = [normalize(s, 'std') for s in d['scn']]
print('after :', test['scn'][0])

### 4.2 Tokenization & subwords
NLLB-200 carries its own SentencePiece tokenizer, so the model side needs no extra
tokenization. For the from-scratch Sockeye baseline we instead port the Napizia tokenizer
(ASCII-fold + uncontract) and bias a small joint BPE toward textbook **desinences**
(*Carinisi are dogs!* → `car@@ in@@ isi are dogs !`), cutting the vocabulary 14k→2k — see
`experiments/tokenization/` and `experiments/baseline/`.

## 5. Model & training

### 5.1 NLLB-200 + LoRA
NLLB-200 is a massively multilingual encoder–decoder that **supports Sicilian** (`scn_Latn`) —
the equivalent of the costly pretraining a from-scratch system would need. We adapt it with
**LoRA**: freeze the weights $W_0$ and learn a low-rank update $W=W_0+\frac{\alpha}{r}BA$
($r{=}32$) on the attention projections — ~1.3% of the parameters, fits a T4.

In [ ]:
model, tok = load_base('facebook/nllb-200-1.3B')
ft = attach_lora(model)

### 5.2 Bidirectional fine-tune (scn↔en)
We train **both directions at once** so one adapter serves both and the weak reverse
direction is rescued (each example is tagged with its forced target language). ~hours on a T4.

In [ ]:
ds = build_dataset(tok, [(train['scn'], train['en'], 'scn', 'en'),
                         (train['en'], train['scn'], 'en', 'scn')])
ft = finetune(ft, tok, ds, out_dir=f'{OUT}/nllb-lora-bidir-1.3B', epochs=2)

## 6. Evaluation (test)

### 6.1 Metrics
We score on the **frozen** literary test set with **sacreBLEU** (BLEU, `tok:13a`) and
**chrF** — chrF being robust for a morphology-rich language. We report both directions.

### 6.2 Main result

In [ ]:
results = {}
for s, t in [('scn', 'en'), ('en', 'scn')]:
    results[f'{s}->{t}'] = score(translate(ft, tok, test[s], s, t), test[t])
    print(f'{s}->{t}:  BLEU={results[f"{s}->{t}"][0]}  chrF={results[f"{s}->{t}"][1]}')
json.dump(results, open(f'{OUT}/results_reproduce.json', 'w'), indent=2)

## 7. Further experiments
We now apply the feasible parts of Wdowiak's recipe on top of NLLB, plus a controlled
ablation. Each is independent and **reloads a fresh model** (heavy; run what you need). The
main result above is already recorded, so these do not affect it.

### 7.1 Italian bridge — one trilingual model
Fine-tune four directions jointly (scn↔en + it↔scn) into a single adapter. Needs
`itsc_{train,test}.{it,scn}` on Drive (frozen WikiMatrix split).

In [ ]:
isc = {s: {l: read(f'{DATA}/itsc_{s}.{l}') for l in ('it', 'scn')} for s in ('train', 'test')}
for s in ('train', 'test'):
    isc[s]['scn'] = [normalize(x, 'std') for x in isc[s]['scn']]
m, tok = load_base('facebook/nllb-200-1.3B'); mft = attach_lora(m)
mft = finetune(mft, tok, build_dataset(tok, [
    (train['scn'], train['en'], 'scn', 'en'), (train['en'], train['scn'], 'en', 'scn'),
    (isc['train']['it'], isc['train']['scn'], 'it', 'scn'),
    (isc['train']['scn'], isc['train']['it'], 'scn', 'it')]),
    out_dir=f'{OUT}/nllb-lora-multi-1.3B', epochs=1)
for s, t, te in [('scn','en',test), ('en','scn',test),
                 ('it','scn',isc['test']), ('scn','it',isc['test'])]:
    print(f'{s}->{t}:', score(translate(mft, tok, te[s], s, t), te[t]))

### 7.2 Back-translation
Back-translate in-domain monolingual English (`mono_en.txt`, harvested from unaligned text)
en→scn with the bidirectional adapter to make synthetic Sicilian, then fine-tune scn→en on
real + synthetic. The English targets stay real, so the decoder learns better English.

In [ ]:
mono = read(f'{DATA}/mono_en.txt')[:15000]
m, tok = load_base('facebook/nllb-200-1.3B', adapter=f'{OUT}/nllb-lora-bidir-1.3B')
synth = translate(m, tok, mono, 'en', 'scn', bs=16, beams=4)
syn, seen = [], set(train['scn'])
for sc, en in zip(synth, mono):
    sc, en = sc.strip(), en.strip()
    if sc and en and sc not in seen and 0.5 <= len(sc.split())/max(1, len(en.split())) <= 2.0:
        seen.add(sc); syn.append((sc, en))
print(f'kept {len(syn)} synthetic pairs')
bft = attach_lora(m)
bft = finetune(bft, tok, build_dataset(tok, [(train['scn'] + [s for s, _ in syn],
               train['en'] + [e for _, e in syn], 'scn', 'en')]),
               out_dir=f'{OUT}/nllb-lora-bt-1.3B', epochs=1)
print('BT scn->en:', score(translate(bft, tok, test['scn'], 'scn', 'en'), test['en']))

### 7.3 Ablation — does normalization help NLLB?
A controlled comparison: three arms (**raw / std / full**), scn→en only, a fresh model each,
**identical English references** (so BLEU is directly comparable). Variants are generated
in-notebook by `normalize_scn`.

In [ ]:
raw = {s: read(f'{DATA}/{s}.scn') for s in ('train', 'test')}   # un-normalized
abl = {}
for arm in ('raw', 'std', 'full'):
    f = (lambda x: x) if arm == 'raw' else (lambda x, a=arm: normalize(x, a))
    m, tok = load_base('facebook/nllb-200-1.3B'); aft = attach_lora(m)
    aft = finetune(aft, tok, build_dataset(tok, [([f(x) for x in raw['train']], train['en'], 'scn', 'en')]),
                   out_dir=f'{OUT}/abl-{arm}', epochs=1)
    abl[arm] = score(translate(aft, tok, [f(x) for x in raw['test']], 'scn', 'en'), test['en'])
    print(f'{arm}:', abl[arm])
print(abl)

## 8. Results & discussion

On our frozen literary test set (scn→en): floor 5.27 → Sockeye levers 10.85 → NLLB-1.3B
zero-shot 29.0 → **bidirectional LoRA 31.4** → + back-translation **31.6**. The reverse
direction en→scn rises 9.9 → **18.7**; Italian is strong (scn→it **43.5**, it→scn **27.0**).

**Method vs data.** Each feasible lever adds little — multilingual ≈0 on scn↔en, normalization
+0.4, back-translation +0.17 (capped by our 7.6k monolingual pool vs his ~550k). So the
**method** gap to Wdowiak's reverse-training state of the art (Sc→En 48.6, En→Sc 45.1, It↔Sc
61–63, on his in-domain test set) is largely closed; the residual is **data** — his
hand-curated corpus, far more back-translation, and an in-domain test set. This is a concrete
argument for combining his corpus with our pipeline rather than competing.

## 9. Serving
The saved adapter backs a local web app and a Telegram bot — one FastAPI `/translate`
endpoint over `experiments/serving/translator.py`. See `experiments/serving/README.md`:
`uvicorn api:app` serves the site at `/`, `telegram_bot.py` runs the bot, and a Docker +
compose deploy scaffold is in the same folder.

## 10. Conclusion
A fully reproducible, Perl-free path from PDFs to a trilingual Sicilian translator reaches
31.6 BLEU (scn→en) — above the published baseline, below the reverse-training SOTA. By
matching the feasible method levers and measuring their small effect, we localize the
remaining gap to **data**, motivating collaboration with the original author.